# 05a — RAG, Approach A: Semantic IDs + multi-stage retrieval

*Two ideas from the literature: TIGER (Semantic IDs) and RAG-FLARKO (multi-stage retrieval).*

**TIGER** (NeurIPS 2023, *Recommender Systems with Generative Retrieval*): each loan's content embedding is **residual-quantised into a hierarchical Semantic ID** (a tuple of codewords). Similar loans share codeword prefixes, so retrieval is *match the prefix, then rank*. We use residual k-means as a light, notebook-friendly stand-in for the paper's RQ-VAE.

**RAG-FLARKO** (Spadea & Seneviratne, 2025): **multi-stage retrieval**. A first *behavioural* stage over the **borrower view** (≈ the Personal KG — who the borrower is) produces a candidate pool; a second *loan/market* stage over the **loan view** (≈ the Market KG — the instrument) re-ranks *within that pool*. Stage 2 is conditioned on Stage 1 (inter-stage context propagation), and only a **compact precedent sub-context** is injected — not the whole corpus.

**Pipeline per loan:** borrower-view Semantic-ID retrieval → behavioural pool (`N_STAGE1`) → loan-view cosine re-rank → top-`K_PRECEDENTS` precedents → inject → LLM decides.

Compared against a **no-RAG LLM** baseline and the **XGBoost** baseline on the same validation loans (`robustness_batch`). The held-out `test_batch` is reserved for Phase 4.

## Setup & config

The first cell pins OpenMP to a single thread, because torch (via sentence-transformers) and XGBoost otherwise load competing runtimes and segfault the kernel on macOS, so it has to run before any other import. The config lines below it set the provider and model, the embedding backend, how many precedents to inject per loan, and `N_TEST`, which caps the run to a handful of loans for a cheap smoke test before paying for the full batch.

In [ ]:
import os
# torch (sentence-transformers) and XGBoost each ship their own OpenMP runtime; on
# macOS loading both in one kernel segfaults. Pin to a single shared OpenMP thread
# BEFORE importing either — this must run before the llm_utils/rag_utils imports below.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import sys; sys.path.insert(0, '..')
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from llm_utils import (
    run_llm_experiment, run_ml_on_sample, evaluate_predictions,
    build_system_prompt, build_user_prompt, load_all_api_keys, RESULTS_DIR,
)
from sample_generation import get_robustness_batch, get_rag_corpus
import rag_utils as R

# ── Config — change these to switch provider / backend ──────────────
API_PROVIDER      = 'openai'                       # GPT-5.4 — the project's chosen model
MODEL_NAME        = 'gpt-5.4'
MODEL_LABEL       = 'GPT-5.4'
EMBEDDING_BACKEND = 'auto'                         # sentence-transformers if installed, else TF-IDF
K_PRECEDENTS      = 8                              # precedents injected per loan
N_STAGE1          = 50                             # stage-1 pool (Approach A)
N_TEST            = None                           # e.g. 50 for a quick smoke run; None = all 100
FORCE_RERUN       = False                          # True ignores the call cache
NOTEBOOK_ID       = '05a_RAG_Generative_SemanticID.ipynb'
WITH_LOGPROBS     = API_PROVIDER in ('openai', 'nvidia', 'gemini')

# Parallel per-loan calls fanned out across every key in .env (same pattern as 01a/01b).
API_KEYS    = load_all_api_keys(API_PROVIDER)
MAX_WORKERS = max(1, len(API_KEYS) * 4)  # len(keys) × 4 workers; lower this if you hit 429s
print(f'{len(API_KEYS)} API key(s) for {API_PROVIDER}; MAX_WORKERS={MAX_WORKERS}')

In [ ]:
# One-time: install the embedding backend if needed (pulls torch).
# Uncomment, run once, then restart the kernel.
# %pip install sentence-transformers
try:
    import sentence_transformers  # noqa: F401
    print('sentence-transformers available')
except Exception:
    print('sentence-transformers NOT installed — Embedder will fall back to TF-IDF.\n'
          'Install it (cell above) for the paper-faithful content embeddings.')

## 1. Corpus + validation set (leakage-checked)
Evaluation runs on the **`robustness_batch`** (validation). The held-out `test_batch` is reserved for Phase 4 and never loaded here.

In [ ]:
corpus = get_rag_corpus()          # all loans EXCEPT the eval/test batches
# Evaluation set = robustness_batch (validation). Strict-holdout protocol:
# test_batch.csv is touched ONLY in Phase 4 (04_Final_Test_Analysis), never here.
test   = get_robustness_batch()        # the 100-row robustness/validation batch
if N_TEST:
    test = test.iloc[:N_TEST].reset_index(drop=True)
R.assert_no_leakage(corpus, test)
print(f'Corpus={len(corpus)}  Eval(robustness)={len(test)}  (no overlap)')

## 2. Embed the two views and build Semantic IDs
Borrower view ≈ Personal KG, loan view ≈ Market KG. Separate embedders keep each view's vector space independent.

In [ ]:
emb_b = R.Embedder(backend=EMBEDDING_BACKEND)
b_corpus = emb_b.fit_transform(R.serialize_frame(corpus, R.BORROWER_FEATURES))
b_test   = emb_b.transform(R.serialize_frame(test,   R.BORROWER_FEATURES))

emb_l = R.Embedder(backend=EMBEDDING_BACKEND)
l_corpus = emb_l.fit_transform(R.serialize_frame(corpus, R.LOAN_FEATURES))
l_test   = emb_l.transform(R.serialize_frame(test,   R.LOAN_FEATURES))

print('embedding backend:', emb_b._resolved, '| borrower', b_corpus.shape, '| loan', l_corpus.shape)

# TIGER Semantic IDs over the borrower (behavioural) view
quant = R.ResidualKMeansQuantizer(n_levels=3, codebook_size=8).fit(b_corpus)
corpus_bsids = quant.semantic_ids(b_corpus)
test_bsids   = quant.semantic_ids(b_test)
print('unique behavioural Semantic IDs:', len(set(corpus_bsids)), '/', len(corpus_bsids))
print('example IDs:', corpus_bsids[:5])

## 3. Multi-stage retrieval → compact precedent context
Stage 1: behavioural candidate pool via Semantic-ID hierarchical retrieval.  
Stage 2: loan-view cosine re-rank within that pool → the injected precedents.

In [ ]:
def retrieve_A(i):
    # Stage 1 (TIGER + behavioural / Personal KG): semantic-ID candidate pool
    pool_idx, _ = R.semantic_id_retrieve(
        b_test[i], test_bsids[i], b_corpus, corpus_bsids, k=N_STAGE1)
    # Stage 2 (loan / Market KG), conditioned on Stage 1: re-rank by loan view
    idx, sims = R.cosine_topk(l_test[i], l_corpus, K_PRECEDENTS, candidate_idx=pool_idx)
    return idx, sims

blocks = []
for i in range(len(test)):
    idx, sims = retrieve_A(i)
    blocks.append(R.build_precedent_block(corpus.iloc[idx], sims=sims))
print(f'Built {len(blocks)} precedent contexts.')
print('\n--- example precedent block (test loan 0) ---\n')
print(blocks[0][:900])

## 4. Run the RAG LLM
Retrieval is precomputed, so the experiment loop just reads each loan's precedent block. ⚠️ This issues one API call per loan — set `N_TEST` small first.

In [ ]:
test_rag, rag_prompt_fn = R.make_rag_prompt_fn(test, blocks)
rag_system_prompt = R.build_rag_system_prompt()

rag = run_llm_experiment(
    test_rag, api_provider=API_PROVIDER, model_name=MODEL_NAME,
    api_keys=API_KEYS, max_workers=MAX_WORKERS, use_cache=not FORCE_RERUN,
    label=f'{MODEL_LABEL} | RAG-A SemanticID', include_desc=False,
    with_logprobs=WITH_LOGPROBS, system_prompt=rag_system_prompt,
    user_prompt_fn=rag_prompt_fn, notebook_id=NOTEBOOK_ID,
    strict=True, max_fail_frac=0.2,
)
print('RAG metrics:', rag['metrics'])

## 5. Baselines — no-RAG LLM and XGBoost (same validation loans)

In [ ]:
# No-RAG LLM control — computed ONCE and recycled across 05a/05b/05c (no re-calls).
# Shared notebook_id unifies the cost log + call-cache across the three notebooks.
base = R.get_norag_baseline(
    test,
    run_fn=lambda: run_llm_experiment(
        test, api_provider=API_PROVIDER, model_name=MODEL_NAME,
        api_keys=API_KEYS, max_workers=MAX_WORKERS, use_cache=not FORCE_RERUN,
        label=f'{MODEL_LABEL} | no-RAG', include_desc=False,
        with_logprobs=WITH_LOGPROBS, notebook_id='05_norag_baseline',
        strict=True, max_fail_frac=0.2,
    ),
    force=FORCE_RERUN,
)

# XGBoost baseline (run_ml_on_sample returns P(Fully Paid), pred via saved threshold)
xgb_probs, xgb_preds = run_ml_on_sample(test)
xgb_metrics = evaluate_predictions(test['loan_status'].values, xgb_preds,
                                   label='XGBoost', probabilities=xgb_probs)

## 6. Compare + save results

In [ ]:
def _row(name, m):
    return {'model': name, **{k: m.get(k) for k in
            ['accuracy', 'precision_charged_off', 'recall_charged_off', 'f1_charged_off', 'auc']}}

summary = pd.DataFrame([
    _row('RAG-A (SemanticID+multistage)', rag['metrics']),
    _row('LLM no-RAG', base['metrics']),
    _row('XGBoost', xgb_metrics),
])
display(summary)
summary.to_csv(f'{RESULTS_DIR}/05a_summary.csv', index=False)

preds = pd.DataFrame({
    'loan_index': range(len(test)),
    'actual':     test['loan_status'].values,
    'rag_pred':   rag['predictions'],
    'rag_prob':   rag['probabilities'],
    'norag_pred': base['predictions'],
    'norag_prob': base['probabilities'],
    'xgb_pred':   xgb_preds,
    'xgb_prob':   xgb_probs,
})
preds.to_csv(f'{RESULTS_DIR}/05a_predictions.csv', index=False)
print(f'Saved -> {RESULTS_DIR}/05a_summary.csv  and  05a_predictions.csv')

In [ ]:
ax = summary.set_index('model')[['accuracy', 'f1_charged_off']].plot.bar(rot=15, figsize=(8, 4))
ax.set_title('Approach A (Semantic ID + multi-stage RAG) vs baselines')
ax.set_ylabel('score'); plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/05a_comparison.png', dpi=150, bbox_inches='tight'); plt.show()